# 40 · Evaluation of the stage-4 runs (additional experiments) - GPU (T4), about 1-2 h, resumable
Read-only for runs and data. Writes `results/p2_stage4/eval/` and the consolidated tables `results/p2_stage4/stage4_*_per_image.csv.gz`,
`stage4_cbis_per_image.csv.gz`, `stage4_weight_change.csv` and `stage4_circuit_codes.npz` (codes entering and leaving the circuit, CBIS-DDSM crops).
* every finished stage-3 run predicts the frozen cohort, DMID, CSAW-S (consensus and the three radiologists) and CDD-CESM crops;
* check: CBIS-DDSM test-fold Dice re-computed for a sample of runs against each run's own `per_image.csv`.

In [ ]:
# 0 · Settings
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json, time, gc, hashlib
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
RES      = f'{PROJECT}/results'
OUT      = f'{RES}/p2_stage4'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
SETS     = {'frozen': f'{PROJECT}/data/processed/cbis_mass_256_jit_frozen', 'dmid': f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi',
            'csaws': f'{PROJECT}/data/external/csaws_mass_256_jit_a1', 'cddcesm': f'{PROJECT}/data/external/cddcesm_mass_256_jit'}
ROOTS    = {'stage4': f'{RES}/p2_stage4_runs_train'}
SCHED    = {'stage4': 'train'}
proto = f'{OUT}/PROTOCOL_stage4_prespecified.txt'
assert os.path.exists(proto), 'protocol missing: run notebook 39 first'
print('protocol', open(proto).read().splitlines()[-1])
missing = [k for k, d in SETS.items() if not os.path.exists(f'{d}/metadata.csv')]
assert not missing, f'{missing} missing'
import glob
QSEG_DIR = f'{PROJECT}/code'                     # qseg package located as in notebooks 13 and 27
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p_ in (QSEG_DIR, f'{PROJECT}/code'):
    if p_ not in sys.path: sys.path.insert(0, p_)
print('qseg package:', QSEG_DIR)
import tensorflow as tf, qseg2, qseg2_stage4; qseg2_stage4.patch()
from qseg.config import Config
cfg = Config(work_dir=OUT); B = cfg.batch_size
print('TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
# 1 · Data
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
DATA = {}
for k, d in SETS.items():
    m = pd.read_csv(f'{d}/metadata.csv')
    X = np.stack([_png(f'{d}/images/{r}.png', False) for r in m.roi_id]); Y = np.stack([_png(f'{d}/masks/{r}.png', True) for r in m.roi_id])
    extra = {}
    for md_ in sorted(glob.glob(f'{d}/masks_*')):                       # per-reader references (CSAW-S)
        extra[os.path.basename(md_)[6:]] = np.stack([_png(f'{md_}/{r}.png', True) for r in m.roi_id])
    DATA[k] = (X, Y, m, extra); print(k, X.shape, '| patients', m.patient.nunique() if 'patient' in m else '-', '| extra refs', list(extra))
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
def load(d, name):
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{d}/best.weights.h5'); return m
def runs_of(root):
    return [d for d in sorted(glob.glob(f'{root}/*_fold*_seed*')) if os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')]
ALL = [(k, d) for k, r in ROOTS.items() for d in runs_of(r)]
print('runs with weights:', len(ALL), '| by seed:', pd.Series([json.load(open(f'{d}/summary.json'))['seed'] for _, d in ALL]).value_counts().sort_index().to_dict())

In [ ]:
# 2 · External sets: every run, every set (resumable)
n = 0
for key, d in ALL:
    todo = [k for k in SETS if not os.path.exists(f'{OUT}/eval/{k}/{key}/{os.path.basename(d)}.csv')]
    if not todo: continue
    s = json.load(open(f'{d}/summary.json')); m = load(d, s['model'])
    for k in todo:
        X, Y, meta, extra = DATA[k]; os.makedirs(f'{OUT}/eval/{k}/{key}', exist_ok=True)
        P = m.predict(X, batch_size=B, verbose=0); t = qseg2.per_image_scores(Y, P, cfg.threshold)
        t.insert(0, 'roi_id', meta.roi_id.values)
        if 'patient' in meta: t.insert(1, 'patient', meta.patient.values)
        t['empty'] = (P[..., 0] >= cfg.threshold).sum((1, 2)) == 0
        for a, Ya in extra.items(): t[f'dice_{a}'] = qseg2.per_image_scores(Ya, P, cfg.threshold).dice.values
        t['set'], t['schedule'], t['model'], t['fold'], t['seed'] = key, SCHED[key], s['model'], int(s['fold']), int(s['seed'])
        t.to_csv(f'{OUT}/eval/{k}/{key}/{os.path.basename(d)}.csv', index=False)
    n += 1; del m; gc.collect()
    if n % 20 == 0: print(time.strftime('%H:%M UTC', time.gmtime()), '| runs evaluated in this session:', n, flush=True)
print('external sets: runs evaluated now', n)

In [ ]:
# 3 · Check: CBIS-DDSM test-fold Dice re-computed for one run per model vs the run's per_image.csv; training progress
done = pd.Series([json.load(open(f'{d}/summary.json'))['model'] for _, d in ALL]).value_counts().sort_index()
print('finished runs by model:', done.to_dict(), '| total', int(done.sum()))
mx = 0.0
for name in sorted(done.index):
    key, d = next((k, x) for k, x in ALL if json.load(open(f'{x}/summary.json'))['model'] == name)
    s = json.load(open(f'{d}/summary.json')); m = load(d, name)
    tr, _, te = qseg2.fold_split(DFC, int(s['fold']))
    P = m.predict(XC[te], batch_size=B, verbose=0); t = qseg2.per_image_scores(YC[te], P, cfg.threshold); t.insert(0, 'roi_id', DFC.roi_id.values[te])
    own = pd.read_csv(f'{d}/per_image.csv').set_index('roi_id').dice
    dd = float(np.abs(t.set_index('roi_id').dice - own.reindex(t.roi_id).values).max()); mx = max(mx, dd)
    print(f'{name:12s} {os.path.basename(d)}: max |dDice| {dd:.2e}'); del m; gc.collect()
print('CBIS re-evaluation check: max |dDice|', mx)

In [ ]:
# 4 · Codes entering and leaving the circuit (CBIS-DDSM crops, all 400) for the wide-encoding circuits (stage 4) and the
#     factor-100 circuits of E1 (stage 3, seeds 0-2) and E1+ (stage 4, seeds 3-4): spread of the encoded angles and additivity
ROOT3 = f'{RES}/p2_stage3_runs_train'
CODES = f'{OUT}/codes'; os.makedirs(CODES, exist_ok=True)
cand = [d for _, d in ALL if json.load(open(f'{d}/summary.json'))['model'] in ('g_q_wide', 'p_q_wide', 'g_q_fast', 'p_q_fast')]
cand += [d for d in runs_of(ROOT3) if json.load(open(f'{d}/summary.json'))['model'] in ('g_q_fast', 'p_q_fast')]
for d in cand:
    f = f'{CODES}/{os.path.basename(os.path.dirname(d))}__{os.path.basename(d)}.npz'
    if os.path.exists(f): continue
    s = json.load(open(f'{d}/summary.json')); m = load(d, s['model']); L = m.get_layer('quantum_circuit')
    sub = tf.keras.Model(m.inputs, [L.input, L.output])
    cin, cout = sub.predict(XC, batch_size=B, verbose=0)
    np.savez_compressed(f, cin=cin.reshape(-1, 4).astype(np.float32), cout=cout.reshape(-1, 4).astype(np.float32),
                        enc=qseg2_stage4.enc_of(s['model']), model=s['model'], fold=int(s['fold']), seed=int(s['seed']))
    del m, sub; gc.collect()
fs = sorted(glob.glob(f'{CODES}/*.npz')); print('code files:', len(fs))
rows = []
for f in fs:
    z = np.load(f); ang = (np.pi / 2) * float(z['enc']) * z['cin']
    rows.append(dict(model=str(z['model']), fold=int(z['fold']), seed=int(z['seed']), angle_sd=float(ang.std(0).mean()),
                     angle_range=float(np.percentile(ang, 97.5) - np.percentile(ang, 2.5))))
A = pd.DataFrame(rows); A.to_csv(f'{OUT}/stage4_angle_spread.csv', index=False)
print(A.groupby('model')[['angle_sd', 'angle_range']].median().round(3).to_string())

In [ ]:
# 5 · Consolidate (full precision, gzip)
for k in list(SETS):
    fs = sorted(glob.glob(f'{OUT}/eval/{k}/*/*.csv'))
    if fs:
        a = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
        a.to_csv(f'{OUT}/stage4_{k}_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
        print(k, a.shape, '| runs', a.groupby(['set', 'model', 'fold', 'seed']).ngroups)
# CBIS-DDSM test folds of all runs (each run's own per_image.csv)
rows = []
for key, d in ALL:
    s = json.load(open(f'{d}/summary.json')); t = pd.read_csv(f'{d}/per_image.csv')
    t['set'], t['schedule'], t['model'], t['fold'], t['seed'] = key, SCHED[key], s['model'], int(s['fold']), int(s['seed']); rows.append(t)
c = pd.concat(rows, ignore_index=True); c.to_csv(f'{OUT}/stage4_cbis_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
print('cbis', c.shape, '| runs', c.groupby(['set', 'model', 'fold', 'seed']).ngroups)
print(c.groupby(['model', 'seed']).dice.mean().unstack().round(4).to_string())
# block weight change and initialization record of every run
W = []
for key, d in ALL:
    s = json.load(open(f'{d}/summary.json'))
    if not os.path.exists(f'{d}/weight_change.json'): continue
    rec = json.load(open(f'{d}/init_record.json')) if os.path.exists(f'{d}/init_record.json') else {}
    for ln, v in json.load(open(f'{d}/weight_change.json')).items():
        W.append(dict(model=s['model'], fold=int(s['fold']), seed=int(s['seed']), layer=ln, epochs_run=s['epochs_run'], best_epoch=s['best_epoch'],
                      shared_identical=rec.get('shared_identical'), slot_scale=rec.get('slot_scale'), encoding_factor=rec.get('encoding_factor', 1.0), **v))
W = pd.DataFrame(W); W.to_csv(f'{OUT}/stage4_weight_change.csv', index=False)
print(W.groupby(['model', 'layer'])[['max_abs_change', 'rel_change']].median().round(4).to_string())
print('DONE')